<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026psy3a_lect06_visual_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 視覚探索課題

このコードは、ブラックボックス化を避けるため、意図的に「色マップ」「方位マップ」「統合マップ」を別々に計算し、それぞれをヒートマップとして視覚化するように設計している。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize_matplotlib
    import japanize_matplotlib

In [ ]:
def simulate_guided_search(target_color, target_ori, noise_level=0.2):
    # 1. 仮想的な視覚ディスプレイの生成 (5x5のグリッド)
    grid_size = 5

    np.random.seed(42) # 再現性のため固定
    colors = np.random.choice([1, -1], size=(grid_size, grid_size))
    orientations = np.random.choice([1, -1], size=(grid_size, grid_size))

    # ターゲットを意図的に配置 (座標)
    target_pos = (2, 2)
    colors[target_pos] = target_color
    orientations[target_pos] = target_ori

    # 2. トップダウン注意の重み設定
    weight_color = target_color
    weight_ori = target_ori

    # 3. 特徴マップの計算
    map_color = colors * weight_color
    map_ori = orientations * weight_ori

    # 4. マスター・アクティベーション・マップの統合
    noise = np.random.normal(0, noise_level, (grid_size, grid_size))
    master_map = map_color + map_ori + noise

    # 5. スポットライト(注意)の移動
    peak_y, peak_x = np.unravel_index(np.argmax(master_map, axis=None), master_map.shape)

    # --- 視覚化 (堅牢な配列処理) ---
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # どのようなレイアウト指定であっても1次元配列に平坦化し、エラーを回避する
    #axes = axes.flatten()

    # 0: 視覚的刺激の代理表示
    axes[0].set_title("Visual Stimulus Array")
    axes[0].imshow(np.zeros((grid_size, grid_size)), cmap='gray', vmin=-2, vmax=2)
    for i in range(grid_size):
        for j in range(grid_size):
            c = 'red' if colors[i,j] == 1 else 'green'
            marker = '|' if orientations[i,j] == 1 else '_'
            axes[0].text(j, i, marker, color=c, fontsize=30, ha='center', va='center', fontweight='bold')

    # 1: 特徴マップ（色）
    axes[1].set_title("特徴地図（色）")
    axes[1].imshow(map_color, cmap='coolwarm', vmin=-2, vmax=2)

    # 2: 特徴マップ（方位）
    axes[2].set_title("特徴地図（方位）")
    axes[2].imshow(map_ori, cmap='coolwarm', vmin=-2, vmax=2)

    # 3: マスターマップとスポットライト
    axes[3].set_title("マスター地図とスポットライト")
    axes[3].imshow(master_map, cmap='viridis')
    circle = plt.Circle((peak_x, peak_y), 0.4, color='white', fill=False, linewidth=3, linestyle='--')
    axes[3].add_patch(circle)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
# 結合探索のシミュレーション (赤 かつ 縦 を探す)
print("【シミュレーション】ターゲット: 「赤」かつ「縦」")
simulate_guided_search(target_color=1, target_ori=1, noise_level=0.5)

1. 上記のコードの `map_color + map_ori` の部分について考察せよ。

「もしターゲットが『色（赤）』だけで区別できる特徴探索だった場合、方位地図のトップダウン重み (weight_ori) を 0 にしたら、マスター地図はどうなるか？」

ノイズに邪魔されることなく、赤色の座標だけが単一の強烈なピークを作る（ポップアウトする）ことを、グラフの色の濃淡から考察せよ。

2. 結合探索における「逐次処理（直列処理）」の発生機序

結合探索（赤・縦）の場合、マスター地図上には「赤・横」や「緑・縦」といった妨害刺激も「中程度のピーク」を作ってしまう。
ここに `noise_level` が加わるとどうなるか？
最高ピーク（白い破線の円）がターゲット（2,2）以外の場所にずれる現象を何度か実行して観察せよ。

「間違ったピークにスポットライトが当たり、違ったから次のピークへ移る」。このピーク間の移動過程こそが、反応時間を遅延させる「直列処理（逐次探索）」の正体である。

3. 半側空間無視への示唆

このモデルの `master_map` を用いて、神経心理学的な症状（半側空間無視検査 BIT の文字抹消課題における左側無視）をどう再現できだろうか。

「脳の損傷によって、空間の左側（座標の x=0, 1 付近）の活性化レベルが強制的に 0.1 倍に減衰してしまうフィルタがマスター地図にかかっていたら、スポットライトはどこを照らすだろうか？

「半側空間無視症例では，視覚的な入力が壊れているのではなく、空間に対する注意の重み付けが歪んでいる，というモデルが構築できるか？を考えよ。